In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
SPLITS = Path("../01_data/interim/splits")
TABLES = Path("../04_outputs/tables")
CHECKPOINTS = Path("../03_models/checkpoints")

TABLES.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
EPSILON = 0.5

source_dataset = "banglasarc3_binary"
target_dataset = "ben_sarc_binary"
EMB_NAME = "word_embeddings"

In [4]:
train_df = pd.read_csv(SPLITS / f"{source_dataset}_train.csv")
val_df = pd.read_csv(SPLITS / f"{source_dataset}_val.csv")
test_df = pd.read_csv(SPLITS / f"{target_dataset}_test.csv")

print("Train source:", train_df.shape)
print("Val source:", val_df.shape)
print("Test target:", test_df.shape)

Train source: (6413, 4)
Val source: (802, 4)
Test target: (2564, 4)


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [7]:
train_df = train_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
val_df = val_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
test_df = test_df[["text", "label_binary"]].rename(columns={"label_binary": "label"})

In [8]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [9]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

Map: 100%|██████████| 2564/2564 [00:00<00:00, 28805.98 examples/s]


In [10]:
train_ds = train_ds.remove_columns(["text"])
val_ds = val_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

In [12]:
class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

In [13]:
class FGMTrainer(Trainer):
    def __init__(self, *args, epsilon=0.5, emb_name="word_embeddings", **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model, epsilon=epsilon, emb_name=emb_name)

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        outputs = model(**inputs)
        loss = outputs.loss
        self.accelerator.backward(loss)

        self.fgm.attack()
        outputs_adv = model(**inputs)
        loss_adv = outputs_adv.loss
        self.accelerator.backward(loss_adv)
        self.fgm.restore()

        return loss.detach() / self.args.gradient_accumulation_steps

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49362.44it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [15]:
training_args = TrainingArguments(
    output_dir=f"../03_models/checkpoints/cross_fgm_{source_dataset}_to_{target_dataset}",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

In [16]:
trainer = FGMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    epsilon=EPSILON,
    emb_name=EMB_NAME,
)

In [17]:
trainer.train()

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Binary,Recall Binary,F1 Binary,Macro F1
1,0.563007,0.499000,0.766833,0.728632,0.850374,0.784810,0.765194
2,0.409779,0.485036,0.774314,0.765700,0.790524,0.777914,0.774255


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.Lay

TrainOutput(global_step=1604, training_loss=0.48639304144424095, metrics={'train_runtime': 1666.1015, 'train_samples_per_second': 7.698, 'train_steps_per_second': 0.963, 'total_flos': 843665599011840.0, 'train_loss': 0.48639304144424095, 'epoch': 2.0})

In [18]:
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = np.array(test_df["label"])

acc = accuracy_score(test_labels, test_preds)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="macro", zero_division=0
)
p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary", zero_division=0
)
cm = confusion_matrix(test_labels, test_preds)

print("Cross-dataset FGM Test Accuracy:", round(acc, 4))
print("Cross-dataset FGM Test Precision (binary):", round(p_bin, 4))
print("Cross-dataset FGM Test Recall (binary):", round(r_bin, 4))
print("Cross-dataset FGM Test F1 (binary):", round(f1_bin, 4))
print("Cross-dataset FGM Test Macro-F1:", round(f1_macro, 4))
print("\nConfusion Matrix:")
print(cm)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Cross-dataset FGM Test Accuracy: 0.6591
Cross-dataset FGM Test Precision (binary): 0.634
Cross-dataset FGM Test Recall (binary): 0.7527
Cross-dataset FGM Test F1 (binary): 0.6883
Cross-dataset FGM Test Macro-F1: 0.6561

Confusion Matrix:
[[725 557]
 [317 965]]


In [19]:
print(classification_report(test_labels, test_preds, zero_division=0))

              precision    recall  f1-score   support

           0       0.70      0.57      0.62      1282
           1       0.63      0.75      0.69      1282

    accuracy                           0.66      2564
   macro avg       0.66      0.66      0.66      2564
weighted avg       0.66      0.66      0.66      2564



In [20]:
results = [
    {
        "model": "banglabert_cross_dataset_fgm",
        "source_dataset": source_dataset,
        "target_dataset": target_dataset,
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "epsilon": EPSILON,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    }
]

results_df = pd.DataFrame(results)
results_df.to_csv(TABLES / f"cross_fgm_{source_dataset}_to_{target_dataset}_results.csv", index=False)

with open(TABLES / f"cross_fgm_{source_dataset}_to_{target_dataset}_confusion_matrix.json", "w", encoding="utf-8") as f:
    json.dump({"confusion_matrix": cm.tolist()}, f, ensure_ascii=False, indent=2)

results_df

,model,source_dataset,target_dataset,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,epochs,batch_size,learning_rate,epsilon,max_length,seed
0,banglabert_cross_dataset_fgm,banglasarc3_binary,ben_sarc_binary,0.659126,0.634034,0.75273,0.688302,0.656113,2,8,0.00002,0.5,128,42
